In [2]:
import pandas as pd
import json
import ollama

print("Biblioteker lastet inn!")


Biblioteker lastet inn!


In [13]:
# De 5 kategoriene vi skal sortere diagnosene i
KATEGORIER = """
1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over
"""

# Last inn rensede data
with open("../data/diagnoses_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Lastet {len(df)} diagnoser")
print(f"\nKategorier:\n{KATEGORIER}")

Lastet 3103 diagnoser

Kategorier:

1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over



In [15]:
def klassifiser_diagnose(diagnose_tekst):
    prompt = f"""Du er en medisinsk historiker som kategoriserer historiske diagnosestrenger.

Kategorier:
{KATEGORIER}

Diagnose: "{diagnose_tekst}"

Svar kun med nummeret (1-5) og kategorinavnet. Eksempel: "3. Ex constitutione"
Ikke skriv noe annet."""

    respons = ollama.chat(
        model="gemma4:12b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return respons["message"]["content"].strip()

# Test med én diagnose
test = klassifiser_diagnose("Hysteria med kramper")
print(f"Testresultat: {test}")

Testresultat: 1. Hysteria


In [6]:
#Plukke opp der den stoppa

import os

TEMP_FILE = "../data/klassifisert_temp.json"

if os.path.exists(TEMP_FILE):
    df_ferdig = pd.read_json(TEMP_FILE)
    ferdig_ids = set(df_ferdig["id"].tolist())
    df_gjenstår = df[~df["id"].isin(ferdig_ids)].copy().reset_index(drop=True)
    print(f"Fortsetter fra temp-fil: {len(df_ferdig)} ferdig, {len(df_gjenstår)} gjenstår")
else:
    df_ferdig = pd.DataFrame()
    df_gjenstår = df.copy().reset_index(drop=True)
    print(f"Starter fra scratch: {len(df_gjenstår)} poster")

Fortsetter fra temp-fil: 2250 ferdig, 853 gjenstår


In [ ]:
#Husk, bytt funksjon i løkka: alle eller gjenstående: 
#Og bytt df.iterrows() til df_gjenstår.iterrows() i løkka.

print("Kategoriserer alle diagnoser...")

resultater = []
for i, (_, rad) in enumerate(df_gjenstår.iterrows()):
    kategori = klassifiser_diagnose(rad["diagnosis_clean"])
    resultater.append(kategori)
    print(f"  [{i+1}/{len(df_gjenstår)}] {rad['id']} → {kategori}")
    
    # Lagre for hver 50. post
    if (i + 1) % 50 == 0:
        df_temp = df_gjenstår.iloc[:i+1].copy()
        df_temp["kategori"] = resultater
        df_combined = pd.concat([df_ferdig, df_temp], ignore_index=True)
        df_combined.to_json("../data/klassifisert_temp.json",
                        orient="records", force_ascii=False, indent=2)
        print(f"  💾 Lagret fremgang ({len(df_combined)} poster totalt)")

# Lagre endelig resultat
df_gjenstår["kategori"] = resultater
df_final = pd.concat([df_ferdig, df_gjenstår], ignore_index=True)
df_final.to_json("../data/klassifisert.json", orient="records", force_ascii=False, indent=2)
print(f"\nFerdig! Totalt {len(df_final)} diagnoser klassifisert.")

In [7]:
# Slå sammen variantene av kategori 1
# Merk: Gemma returnerte av og til ekstra tekst i kategori 1.
# Dette normaliseres her for konsistent kategorisering.

df_ferdig["kategori"] = df_ferdig["kategori"].str.replace(
    r"^1\. Hysteria.*", "1. Hysteria", regex=True
)
df_ferdig.to_json("../data/klassifisert.json", orient="records", force_ascii=False, indent=2)
print("Fikset! Ny fordeling:")
print(df_ferdig["kategori"].value_counts())

Fikset! Ny fordeling:
kategori
1. Hysteria                  1583
3. Ex constitutione          1133
2. Hysterisk personlighet     203
5. Annet                      134
4. Epilepsi-relatert           50
Name: count, dtype: int64


In [ ]:
#Sjekk at alle data er lagret

import pandas as pd

df_ferdig = pd.read_json("../data/klassifisert.json")
print(f"Antall klassifiserte: {len(df_ferdig)}")
print(f"\nKolonner: {df_ferdig.columns.tolist()}")
print(f"\nFordeling per kategori:")
print(df_ferdig["kategori"].value_counts())
df_ferdig.head()